# RAG-Based Financial Guidance Chatbot — MVP
**CS 4650 Final Project** · Ronit, Sharada, Akshaya

This notebook implements the full RAG pipeline end-to-end:
1. Load FiQA dataset
2. Chunk + embed documents
3. Build FAISS vector index
4. Retrieval (dense) + BM25 baseline
5. Generation via HuggingFace Inference API
6. Evaluation: Recall@k, BERTScore, no-retrieval baseline

**Runtime:** Colab with T4 GPU recommended (Runtime → Change runtime type → T4 GPU)

## 0. Setup

In [ ]:
!pip install -U transformers bert-score sentence-transformers faiss-cpu datasets rank-bm25

  Using cached transformers-5.7.0-py3-none-any.whl.metadata (33 kB)
  Using cached bert_score-0.3.13-py3-none-any.whl.metadata (15 kB)
Using cached transformers-5.7.0-py3-none-any.whl (10.5 MB)
Using cached bert_score-0.3.13-py3-none-any.whl (61 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 67.5 MB/s eta 0:00:00


In [ ]:
!pip uninstall -y transformers tokenizers bert-score

Found existing installation: transformers 5.7.0
Uninstalling transformers-5.7.0:
  Successfully uninstalled transformers-5.7.0
Found existing installation: tokenizers 0.22.2
Uninstalling tokenizers-0.22.2:
  Successfully uninstalled tokenizers-0.22.2
Found existing installation: bert-score 0.3.13
Uninstalling bert-score-0.3.13:
  Successfully uninstalled bert-score-0.3.13


In [ ]:
import os
import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm

# Set your HF token here (get one free at https://huggingface.co/settings/tokens)
# Use a 'read' token. Store in Colab secrets for safety.
from google.colab import userdata
try:
    HF_TOKEN = userdata.get('HF_TOKEN')
except Exception:
    HF_TOKEN = input('Paste HF token: ').strip()
os.environ['HF_TOKEN'] = HF_TOKEN

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)

## 1. Load FiQA dataset

FiQA-2018 has two tasks. We use **Task 2 (opinion Q&A)** which gives us:
- A corpus of financial forum posts (the "documents" we retrieve from)
- A set of queries
- Relevance judgments (query → which docs are relevant) — essential for Recall@k

We pull the BEIR version because it has clean train/dev/test splits and qrels.

In [ ]:
from datasets import load_dataset

# BEIR's FiQA — corpus, queries, qrels
corpus_ds = load_dataset('BeIR/fiqa', 'corpus')['corpus']
queries_ds = load_dataset('BeIR/fiqa', 'queries')['queries']
qrels_ds = load_dataset('BeIR/fiqa-qrels')

print(f'Corpus: {len(corpus_ds):,} docs')
print(f'Queries: {len(queries_ds):,}')
print('Qrels splits:', list(qrels_ds.keys()))
print('Example doc:', corpus_ds[0])
print('Example query:', queries_ds[0])
print('Example qrel:', qrels_ds['test'][0])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

corpus/corpus-00000-of-00001.parquet:   0%|          | 0.00/27.7M [00:00<?, ?B/s]

Generating corpus split:   0%|          | 0/57638 [00:00<?, ? examples/s]

queries/queries-00000-of-00001.parquet:   0%|          | 0.00/322k [00:00<?, ?B/s]

...

test.tsv: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/14166 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1238 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1706 [00:00<?, ? examples/s]

Corpus: 57,638 docs
Queries: 6,648
Qrels splits: ['train', 'validation', 'test']
Example doc: {'_id': '3', 'title': '', 'text': "I'm not saying I don't like the idea of on-the-job training too, but you can't expect the company to do that. Training workers is not their job - they're building software. Perhaps educational systems in the U.S. (or their students) should worry a little about getting marketable skills in exchange for their massive investment in education, rather than getting out with thousands in student debt and then complaining that they aren't qualified to do anything."}
Example query: {'_id': '0', 'title': '', 'text': 'What is considered a business expense on a business trip?'}
Example qrel: {'query-id': 8, 'corpus-id': 566392, 'score': 1}


In [ ]:
# Convert to convenient dicts
corpus = {row['_id']: (row.get('title','') + ' ' + row['text']).strip() for row in corpus_ds}
queries = {row['_id']: row['text'] for row in queries_ds}

# qrels: {query_id: {doc_id: relevance}}
from collections import defaultdict
qrels_test = defaultdict(dict)
for row in qrels_ds['test']:
    qrels_test[row['query-id']][row['corpus-id']] = int(row['score'])

# Only keep queries that have qrels in test split
test_query_ids = list(qrels_test.keys())
print(f'Test queries with qrels: {len(test_query_ids)}')

# Sanity: sample a query + its relevant docs
qid = test_query_ids[1]


print(f'\nQuery [{qid}]: {queries[str(qid)]}')
for did in list(qrels_test[str(qid)].keys())[:2]:
    print(f'\nRelevant doc [{did}]: {corpus[did][:300]}...')

Test queries with qrels: 648

Query [15]: Can I send a money order from USPS as a business?


## 2. Chunk documents

FiQA docs are already short (forum posts), so chunking is light. We split anything over ~400 words to avoid embedding truncation.

In [ ]:
def chunk_text(text, max_words=400, overlap=50):
    words = text.split()
    if len(words) <= max_words:
        return [text]
    chunks = []
    step = max_words - overlap
    for i in range(0, len(words), step):
        chunk = ' '.join(words[i:i+max_words])
        chunks.append(chunk)
        if i + max_words >= len(words):
            break
    return chunks

# Build chunk-level corpus. Each chunk keeps a pointer to its parent doc_id.
chunks = []  # list of (chunk_id, parent_doc_id, text)
for did, text in tqdm(corpus.items(), desc='Chunking'):
    for i, c in enumerate(chunk_text(text)):
        chunks.append((f'{did}__{i}', did, c))

print(f'{len(corpus):,} docs → {len(chunks):,} chunks')
chunk_texts = [c[2] for c in chunks]
chunk_parents = [c[1] for c in chunks]

Chunking:   0%|          | 0/57638 [00:00<?, ?it/s]

57,638 docs → 60,395 chunks


## 3. Embed chunks with sentence-transformers

`BAAI/bge-small-en-v1.5` — small (33M params), fast on free Colab, strong on BEIR.

In [ ]:
from sentence_transformers import SentenceTransformer

EMBED_MODEL = 'BAAI/bge-small-en-v1.5'
embedder = SentenceTransformer(EMBED_MODEL, device=DEVICE)
print('Embedding dim:', embedder.get_sentence_embedding_dimension())

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding dim: 384


/tmp/ipykernel_2048/2035146028.py:5: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print('Embedding dim:', embedder.get_sentence_embedding_dimension())


In [ ]:

# Encode corpus (~57k chunks — takes 3-5 min on T4)
corpus_embs = embedder.encode(
    chunk_texts,
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,  # cosine via inner product on normalized vectors
)
print('Corpus embeddings:', corpus_embs.shape)

NameError: name 'chunk_texts' is not defined

In [ ]:
# Save so you don't have to re-embed
# np.save('corpus_embs.npy', corpus_embs)
# Also save chunk metadata
import json
with open('chunks_meta.json', 'w') as f:
    json.dump([{'chunk_id': c[0], 'parent': c[1]} for c in chunks], f)

## 4. Build FAISS index

In [ ]:
import faiss

corpus_embs = np.load('corpus_embs.npy')

dim = corpus_embs.shape[1]
index = faiss.IndexFlatIP(dim)  # inner product = cosine (embeddings are normalized)
index.add(corpus_embs.astype('float32'))
print('FAISS index size:', index.ntotal)

FAISS index size: 60395


## 5. BM25 baseline (for retrieval comparison)

In [ ]:
from rank_bm25 import BM25Okapi
import re

def tokenize(t):
    return re.findall(r'\w+', t.lower())

tokenized_chunks = [tokenize(t) for t in tqdm(chunk_texts, desc='Tokenizing')]
bm25 = BM25Okapi(tokenized_chunks)
print('BM25 ready')

Tokenizing:   0%|          | 0/60395 [00:00<?, ?it/s]

BM25 ready


## 6. Retrieval functions

In [ ]:
def retrieve_dense(query, k=5):
    q_emb = embedder.encode([query], normalize_embeddings=True, convert_to_numpy=True).astype('float32')
    scores, idxs = index.search(q_emb, k)
    results = []
    for s, i in zip(scores[0], idxs[0]):
        results.append({
            'chunk_id': chunks[i][0],
            'doc_id': chunks[i][1],
            'text': chunks[i][2],
            'score': float(s),
        })
    return results

def retrieve_bm25(query, k=5):
    q_tok = tokenize(query)
    scores = bm25.get_scores(q_tok)
    top = np.argsort(scores)[::-1][:k]
    return [{
        'chunk_id': chunks[i][0],
        'doc_id': chunks[i][1],
        'text': chunks[i][2],
        'score': float(scores[i]),
    } for i in top]

# Smoke test
q = 'What are the risks of investing in penny stocks?'
print('=== Dense ===')
for r in retrieve_dense(q, k=3):
    print(f'[{r["score"]:.3f}] {r["text"][:150]}...')
print('\n=== BM25 ===')
for r in retrieve_bm25(q, k=3):
    print(f'[{r["score"]:.3f}] {r["text"][:150]}...')

=== Dense ===
[0.819] The biggest problem with penny stocks is that they are easily manipulated, and they frequently are.  Many of the companies trading as penny stocks hav...
[0.814] Shorting penny stocks is very risky.  For example, read this investopedia article, which explains some of the problems.  In general: If you have some ...
[0.814] Penny stocks are only appealing to two types of investors: Most of the beginners who invest in penny stocks only do so because they don't have a lot o...

=== BM25 ===
[38.675] Penny stocks are only appealing to two types of investors: Most of the beginners who invest in penny stocks only do so because they don't have a lot o...
[33.508] Is it safe to invest in a portfolio of dividend stocks yielding 7-9% with the money borrowed at 3-4% from one of these brokerages? Yes and no. It depe...
[32.988] The biggest problem with penny stocks is that they are easily manipulated, and they frequently are.  Many of the companies trading as penny stocks hav.

## 7. Generation via HuggingFace Inference API

Free tier. We use a small instruct model — swap the model id if you want something bigger.

In [ ]:
from huggingface_hub import InferenceClient

# Options to try:
#   'mistralai/Mistral-7B-Instruct-v0.3'
#   'meta-llama/Llama-3.2-3B-Instruct'
#   'HuggingFaceH4/zephyr-7b-beta'
GEN_MODEL = 'meta-llama/Meta-Llama-3-8B-Instruct'
client = InferenceClient(model=GEN_MODEL, token=HF_TOKEN)

SYSTEM_PROMPT = (
    'You are a helpful financial-guidance assistant. Answer the user\'s question '
    'using ONLY the provided context. If the context does not contain enough information, '
    'say so honestly rather than guessing. Be concise (2-4 sentences).'
)

def build_user_content(question, retrieved_docs):
    context = '\n\n'.join([f'[{i+1}] {r["text"]}' for i, r in enumerate(retrieved_docs)])
    return f'Context:\n{context}\n\nQuestion: {question}'

def generate(question, retrieved_docs=None, max_new_tokens=200):
    if retrieved_docs:
        messages = [
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user', 'content': build_user_content(question, retrieved_docs)},
        ]
    else:
        messages = [
            {'role': 'system', 'content': 'You are a helpful financial-guidance assistant. Be concise (2-4 sentences).'},
            {'role': 'user', 'content': question},
        ]

    prompt = SYSTEM_PROMPT,

    resp = client.chat_completion(messages = messages,
    model="meta-llama/Meta-Llama-3-8B-Instruct",
    max_tokens=300
    )
    return resp.choices[0].message.content.strip()

# Smoke test
q = 'What are the risks of investing in penny stocks?'
docs = retrieve_dense(q, k=3)
print('=== With RAG ===')
print(generate(q, docs))
print('\n=== No retrieval ===')
print(generate(q, None))

=== With RAG ===
Investing in penny stocks comes with several significant risks, including easily manipulated companies, poor financial track histories, and unreliable information. Additionally, liquidity issues can make it difficult to liquidate positions, and the lack of competitive market makers can lead to evaporating margins.

=== No retrieval ===
Investing in penny stocks carries significant risks, including high volatility, lack of liquidity, and potential for price manipulation. Penny stocks are often thinly traded, making it difficult to sell shares quickly, and prices can fluctuate rapidly due to limited market data. Additionally, many penny stocks are issued by companies with questionable financials or unproven business models, increasing the likelihood of investment losses.


## 8. Evaluation

### 8a. Retrieval: Recall@k

For each test query, check if any relevant doc (from qrels) appears in top-k retrieved.

**Note on chunk↔doc mapping:** we retrieve chunks but qrels are on parent docs. A chunk counts as a hit if its `parent` is in qrels.

In [ ]:
def recall_at_k(retrieve_fn, test_qids, k_values=(1, 5, 10, 20)):
    hits = {k: 0 for k in k_values}

    for qid in tqdm(test_qids, desc=f'Eval {retrieve_fn.__name__}'):
        relevant = set([str(k) for k in qrels_test[qid].keys()])
        if not relevant:
            continue
        retrieved = retrieve_fn(queries[str(qid)], k=max(k_values))
        retrieved_docs = [r['doc_id'] for r in retrieved]
        # Dedupe (multiple chunks → same parent)
        seen = []
        for d in retrieved_docs:
            if d not in seen:
                seen.append(d)
        for k in k_values:
            topk = set(seen[:k])
            if topk & relevant:
                hits[k] += 1
    return {k: hits[k] / len(test_qids) for k in k_values}

# Subsample for speed — use all 648 for final numbers
SAMPLE = test_query_ids[:648]
print('Dense:', recall_at_k(retrieve_dense, SAMPLE))
print('BM25: ', recall_at_k(retrieve_bm25,  SAMPLE))

Eval retrieve_dense:   0%|          | 0/648 [00:00<?, ?it/s]

Dense: {1: 0.38425925925925924, 5: 0.5725308641975309, 10: 0.6358024691358025, 20: 0.7021604938271605}


Eval retrieve_bm25:   0%|          | 0/648 [00:00<?, ?it/s]

BM25:  {1: 0.19907407407407407, 5: 0.35802469135802467, 10: 0.4398148148148148, 20: 0.5262345679012346}


### 8b. Generation: BERTScore vs. no-retrieval baseline

FiQA doesn't have a single gold answer per query, but each query's relevant docs are user-accepted forum answers. We treat the concatenated relevant-doc text as reference.

In [ ]:
from bert_score import score as bertscore


# Small eval batch (generation is slow via API)
EVAL_N = 30
eval_qids = test_query_ids[:EVAL_N]

predictions_rag_dense = []
predictions_no_retrieval = []
predictions_rag_bm25 = []
references = []

for qid in tqdm(eval_qids, desc='Generating'):
    q = queries[str(qid)]
    rel_docs = list(qrels_test[qid].keys())
    ref = ' '.join(corpus[d] for d in rel_docs if d in corpus)[:2000]

    # Ensure reference isn't empty either
    references.append(ref if ref.strip() else ".")

    try:
        res = generate(q, retrieve_dense(q, k=5))
        predictions_rag_dense.append(res if res.strip() else ".")
    except Exception as e:
        predictions_rag_dense.append(".") # Fix: Use "." instead of ""
        print(f'RAG err on {qid}: {e}')

    try:
        res = generate(q, retrieve_dense(q, k=5))
        predictions_rag_bm25.append(res if res.strip() else ".")
    except Exception as e:
        predictions_rag_bm25.append(".") # Fix: Use "." instead of ""
        print(f'RAG err on {qid}: {e}')

    try:
        res_no = generate(q, None)
        predictions_no_retrieval.append(res_no if res_no.strip() else ".")
    except Exception as e:
        predictions_no_retrieval.append(".") # Fix: Use "." instead of ""
        print(f'No-RAG err on {qid}: {e}')



Generating:   0%|          | 0/30 [00:00<?, ?it/s]

In [ ]:
P_rag_dense, R_rag_dense, F_rag_dense = bertscore(predictions_rag_dense, references, lang='en', model_type = "roberta-large", verbose=False, use_fast_tokenizer=True)
P_rag_bm, R_rag_bm, F_rag_bm = bertscore(predictions_rag_bm25, references, lang='en', model_type = "roberta-large", verbose=False, use_fast_tokenizer=True)

P_nr,  R_nr,  F_nr  = bertscore(predictions_no_retrieval, references, lang='en', model_type = "roberta-large", verbose=False, use_fast_tokenizer=True)

print(f'RAG Dense        BERTScore-F1: {F_rag_dense.mean().item():.4f}')
print(f'RAG BM25       BERTScore-F1: {F_rag_bm.mean().item():.4f}')
print(f'No retrieval BERTScore-F1: {F_nr.mean().item():.4f}')

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


RAG Dense        BERTScore-F1: 0.8973
RAG BM25       BERTScore-F1: 0.8973
No retrieval BERTScore-F1: 0.8898


### 8c. Save outputs for qualitative analysis / human eval

In [ ]:
eval_df = pd.DataFrame({
    'qid': eval_qids,
    'question': [queries[str(q)] for q in eval_qids],
    'rag_answer': predictions_rag_dense,
    'no_retrieval_answer': predictions_no_retrieval,
    'reference_snippet': [r[:500] for r in references],
})
eval_df.to_csv('eval_outputs.csv', index=False)
eval_df.head()

,qid,question,rag_answer,no_retrieval_answer,reference_snippet
0,8,How to deposit a cheque issued to an associate...,You can deposit the cheque into your business ...,You can deposit a cheque issued to your associ...,.
1,15,Can I send a money order from USPS as a business?,You can fill in your business name and address...,"Yes, as a business, you can purchase and send ...",.
2,18,1 EIN doing business under multiple business n...,You can use one EIN to do business under multi...,If one Employer Identification Number (EIN) is...,.
3,26,Applying for and receiving business credit,"To apply for business credit, you'll typically...","To apply for business credit, you'll need to e...",.
4,34,401k Transfer After Business Closure,"When a business closes, the 401(k) transfer pr...","When a business closes, it's essential to unde...",.


## 9. Interactive chatbot

In [ ]:
def chat(question, k=5, retriever='dense', show_sources=True):
    retrieve_fn = retrieve_dense if retriever == 'dense' else retrieve_bm25
    docs = retrieve_fn(question, k=k)
    answer = generate(question, docs)
    print(f'Q: {question}\n')
    print(f'A: {answer}\n')
    if show_sources:
        print('--- Sources ---')
        for i, d in enumerate(docs[:3]):
            print(f'[{i+1}] (score={d["score"]:.3f}) {d["text"][:200]}...\n')

chat('How does a 401k differ from a Roth IRA?')

Q: How does a 401k differ from a Roth IRA?

A: A 401(k) and a Roth IRA differ in their tax treatment. With a 401(k), the contributions are tax-deferred, meaning you don't pay taxes on them now, but you'll pay taxes on withdrawals in retirement. In contrast, a Roth IRA contribution is made with after-tax money, but withdrawals are tax-free.

--- Sources ---
[1] (score=0.854) "If I understand correctly, the Traditional IRA, if you have 401k with an employer already, has the following features: Actually, #1 and #2 are characteristics of Roth IRAs, not Traditional IRAs. Only...

[2] (score=0.852) You can contribute to a Traditional IRA instead of a Roth.  The main difference is a contribution to a Roth is made with after tax money but at retirement you can withdraw the money tax free.  With a ...

[3] (score=0.846) "I am failing to see why would a person get an IRA, instead of just putting the same amount of money into a mutual fund (like Vanguard) or something like that. Well, this isn't 